# Concatenate CSV and JSONL files

Reads a series of `.csv` and `.jsonl` files from an input directory and concatenates them, **separately by format**, into a single `.csv` and a single `.jsonl` file.

Configure the input/output paths and the glob patterns in the **Configuration** cell below, then run all cells.

In [1]:
from pathlib import Path
import json
import csv
import re
import sys
import shutil

# Some cells (e.g. long generated text) exceed the default 128 KB CSV field limit.
# Raise it to the largest value the platform accepts.
_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(_limit)
        break
    except OverflowError:
        _limit //= 2

## Configuration

In [13]:
# Directory containing the input files (relative to the repo root).
INPUT_DIR = Path("../out_data")

# Files are selected by extension + a regex that must match (INCLUDE) and an
# optional regex that must NOT match (EXCLUDE). Both regexes use re.search, so a
# bare substring like "final_results" works, but full regex syntax is available.
# Set EXCLUDE to None to disable exclusion.
INCLUDE = r"final_results_v3_[0-9]+_.+"
EXCLUDE = r"vanilla"

# Output files.
OUTPUT_DIR = Path("../out_data")
CSV_OUTPUT = OUTPUT_DIR / "concatenated.csv"
JSONL_OUTPUT = OUTPUT_DIR / "concatenated.jsonl"

# Sort input files by name for deterministic ordering.
SORT_INPUTS = True

In [14]:
def collect_files(
    directory: Path,
    extension: str,
    include: str = INCLUDE,
    exclude: str | None = EXCLUDE,
    output: Path | None = None,
) -> list[Path]:
    """Return files in ``directory`` whose name matches ``include`` but not ``exclude``.

    Selection is: right ``extension`` AND ``include`` matches AND ``exclude`` does
    not match (when given). The ``output`` file is always excluded so re-running
    never folds a previous result back in.
    """
    include_re = re.compile(include)
    exclude_re = re.compile(exclude) if exclude else None
    output = output.resolve() if output else None

    files = [
        path
        for path in directory.glob(f"*{extension}")
        if include_re.search(path.name)
        and not (exclude_re and exclude_re.search(path.name))
        and path.resolve() != output
    ]
    return sorted(files) if SORT_INPUTS else files


csv_files = collect_files(INPUT_DIR, ".csv", output=CSV_OUTPUT)
jsonl_files = collect_files(INPUT_DIR, ".jsonl", output=JSONL_OUTPUT)

print(f"CSV files ({len(csv_files)}):")
for f in csv_files:
    print(f"  {f.name}")
print(f"\nJSONL files ({len(jsonl_files)}):")
for f in jsonl_files:
    print(f"  {f.name}")

CSV files (3):
  final_results_v3_0612_results.csv
  final_results_v3_0615_results_0.csv
  final_results_v3_0615_results_1.csv

JSONL files (3):
  final_results_v3_0612_details.jsonl
  final_results_v3_0615_details_0.jsonl
  final_results_v3_0615_details_1.jsonl


## Concatenate CSV files

The header from the first file is written once; headers from subsequent files are skipped. Files are streamed line by line so very large files don't need to fit in memory. A mismatched header in a later file raises an error rather than silently corrupting the output.

In [15]:
def concatenate_csv(files: list[Path], output: Path) -> int:
    """Concatenate CSV ``files`` into ``output``, keeping a single header row."""
    if not files:
        print("No CSV files to concatenate.")
        return 0

    output.parent.mkdir(parents=True, exist_ok=True)
    header = None
    rows_written = 0

    with open(output, "w", newline="", encoding="utf-8") as out_f:
        writer = csv.writer(out_f)
        for path in files:
            with open(path, newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                try:
                    file_header = next(reader)
                except StopIteration:
                    print(f"  skipping empty file: {path.name}")
                    continue
                if header is None:
                    header = file_header
                    writer.writerow(header)
                elif file_header != header:
                    raise ValueError(
                        f"Header mismatch in {path.name}.\n"
                        f"  expected: {header}\n"
                        f"  found:    {file_header}"
                    )
                file_rows = 0
                for row in reader:
                    writer.writerow(row)
                    file_rows += 1
                rows_written += file_rows
                print(f"  {path.name}: {file_rows} rows")

    print(f"\nWrote {rows_written} data rows to {output}")
    return rows_written


concatenate_csv(csv_files, CSV_OUTPUT)

  final_results_v3_0612_results.csv: 399 rows
  final_results_v3_0615_results_0.csv: 399 rows
  final_results_v3_0615_results_1.csv: 399 rows

Wrote 1197 data rows to ../out_data/concatenated.csv


1197

## Concatenate JSONL files

Each line is validated as JSON and re-emitted as a single compact line. Blank lines are skipped. Files are streamed line by line.

In [16]:
def concatenate_jsonl(files: list[Path], output: Path) -> int:
    """Concatenate JSONL ``files`` into ``output``, validating one record per line."""
    if not files:
        print("No JSONL files to concatenate.")
        return 0

    output.parent.mkdir(parents=True, exist_ok=True)
    records_written = 0

    with open(output, "w", encoding="utf-8") as out_f:
        for path in files:
            file_records = 0
            with open(path, encoding="utf-8") as in_f:
                for lineno, line in enumerate(in_f, start=1):
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError as exc:
                        raise ValueError(
                            f"Invalid JSON in {path.name} line {lineno}: {exc}"
                        ) from exc
                    out_f.write(json.dumps(record, ensure_ascii=False))
                    out_f.write("\n")
                    file_records += 1
            records_written += file_records
            print(f"  {path.name}: {file_records} records")

    print(f"\nWrote {records_written} records to {output}")
    return records_written


concatenate_jsonl(jsonl_files, JSONL_OUTPUT)

  final_results_v3_0612_details.jsonl: 399 records
  final_results_v3_0615_details_0.jsonl: 399 records
  final_results_v3_0615_details_1.jsonl: 399 records

Wrote 1197 records to ../out_data/concatenated.jsonl


1197

In [12]:
import pandas as pd

In [12]:
df = pd.read_csv("/Users/bastianellie/Library/CloudStorage/OneDrive-SharedLibraries-ReedElsevierGroupICOReedElsevierInc/OG-Data Science Research Content - Researcher AI/evaluation/writing-coach/q1-beta-release/benchmark/writing-coach-query-inputtext-natural+synth.csv")

In [29]:
lines = [
5, 93, 94, 95, 96, 130, 150, 151, 152, 153, 154, 155, 156, 157, 173, 174, 175, 176, 177, 178, 180, 181, 183, 184, 186, 187, 192, 199, 200, 201, 202, 203, 204, 205, 207, 208, 209, 213, 215, 216, 217, 220, 224, 227, 228, 229, 230, 231, 233, 236, 237, 238, 240, 241, 242, 243, 245, 246, 248
]

In [30]:
for l in lines:
    text = df.iloc[l].input
    print("-" * 24, l, "-" * 24)
    print(text)

------------------------ 5 ------------------------
Modern cities have become a key element of the strategies for future investments not only for the important role they play in the socio-economic growth worldwide, but also for the increasing difficulty in being environmentally sustainable [31, 71]. As a consequence, a considerable number of efforts have been put by both the academic and industrial research communities into exploiting techniques from a wide range of domains such as ubiquitous computing, data analysis and knowledge engineering, with the aim of enabling cities to be more efficient, productive and competitive [4, 9, 24, 52, 54].\n\nRobotics has also reached increased maturity in the last decades, as demonstrated by the amount of efficient open source hardware and software components, the availability of reliable techniques for basic perception, manipulation and navigation tasks, as well as the increasing number of cost-accessible robotic platforms [82]. Thanks to these ad